## Experiment 1
Date Created: September 5, 2025 \
Last Edited: October 12, 2025

### Environment Setup

In [2]:
import osmnx as ox 
import numpy as np 
import folium
import geopandas as gpd
from shapely.geometry import LineString
from pathlib import Path 
import random

### Extract OSM Data

In [6]:
def extract_osm_data(location, radius=500):
    """
    Extracts building footprints and road networks from OpenStreetMap data for a given location.
    """

    print(f"Extracting OSM data for {location} within a radius of {radius} meters...")

    print("Configuring OSMnx settings...")
    # Set Overpass endpoint
    ox.settings.overpass_endpoint = "https://overpass.kumi.systems/api/interpreter"
    ox.settings.timeout = 180

    # Get latitude and longitude of the location
    print("Geocoding the location...")
    center_point = ox.geocoder.geocode(location)

    # Download walkable areas footprints 
    print("Downloading walkable areas...")
    streets = ox.graph_from_point(
        center_point,
        dist=radius,
        network_type='walk',
        simplify=False
    )

    # Convert to GeoDataFrames
    print("Converting to GeoDataFrames...")
    # buildings = buildings.to_crs(epsg=2154)
    streets = ox.project_graph(streets, to_crs='epsg:2154')

    print(f"Extracted {len(streets.edges)} street segments.")

    return center_point, streets

In [7]:
center_point, streets = extract_osm_data("Mandaue City, Philippines", radius=1000)

Extracting OSM data for Mandaue City, Philippines within a radius of 1000 meters...
Configuring OSMnx settings...
Geocoding the location...
Converting to GeoDataFrames...
Extracted 2836 street segments.


### Generate map data

In [8]:
# Get centroids for folium map init
center_lat, center_lon = center_point[0], center_point[1]

# Generate folium map 
m = folium.Map(location=[center_lat, center_lon], tiles='CartoDB dark_matter', zoom_start=16)

# Convert streets graph to GeoDataFrame
edges = ox.graph_to_gdfs(streets, nodes=False, edges=True)

# Add streets
folium.GeoJson(edges.to_crs(epsg=4326), 
               style_function=lambda x: {"color": "#FFDF22", "weight": 3}
              ).add_to(m)

m.save("osm_map.html")
m